# Fraud Detection with BigQuery AI Functions

This notebook demonstrates how to use BigQuery AI functions (Gemini integration) to process unstructured data (receipt images) and combine it with structured transaction data for fraud detection.

## Prerequisites
- External connection `gemini-connection` created in location `US`.
- IAM roles granted to the connection's service account:
    - `roles/aiplatform.user`
    - `roles/storage.objectViewer` on `gs://fraud-prevention-demo-receipts`

In [ ]:
# Create a remote model pointing to Gemini
CREATE OR REPLACE MODEL `fraud_data.gemini_model`
REMOTE WITH CONNECTION `fraud-prevention-demo.US.gemini-connection`
OPTIONS(ENDPOINT = 'gemini-1.5-flash');

In [ ]:
# Create an object table for receipts in GCS
CREATE OR REPLACE EXTERNAL TABLE `fraud_data.receipts_object_table`
WITH CONNECTION `fraud-prevention-demo.US.gemini-connection`
OPTIONS (
  object_metadata = 'SIMPLE',
  uris = ['gs://fraud-prevention-demo-receipts/*']
);

In [ ]:
# Extract data from receipts using Gemini
SELECT
  uri,
  ml_generate_text_result
FROM
  ML.GENERATE_TEXT(
    MODEL `fraud_data.gemini_model`,
    TABLE `fraud_data.receipts_object_table`,
    STRUCT(
      'Analyze this receipt image and extract the following fields in JSON format: total_amount, vendor_name, date. Do not include any markdown formatting in the response.' AS prompt,
      TRUE AS flatten_json_output
    )
  );

In [ ]:
# Join extracted data with structured transactions
-- Note: We are using historical_transactions for this example,
-- but you can use streaming_transactions for real-time analysis.

WITH ExtractedData AS (
  SELECT
    uri,
    -- Extracting fields from the JSON result
    SAFE_CAST(JSON_EXTRACT_SCALAR(ml_generate_text_result, '$.total_amount') AS FLOAT64) AS extracted_amount,
    JSON_EXTRACT_SCALAR(ml_generate_text_result, '$.vendor_name') AS extracted_vendor,
    JSON_EXTRACT_SCALAR(ml_generate_text_result, '$.date') AS extracted_date
  FROM
    ML.GENERATE_TEXT(
      MODEL `fraud_data.gemini_model`,
      TABLE `fraud_data.receipts_object_table`,
      STRUCT(
        'Analyze this receipt image and extract the following fields in JSON format: total_amount, vendor_name, date.' AS prompt
      )
    )
)
SELECT
  t.transaction_id,
  t.amount,
  e.extracted_amount,
  t.merchant,
  e.extracted_vendor,
  t.event_timestamp,
  e.extracted_date,
  -- Simple fraud indicator: amount mismatch
  IF(ABS(t.amount - e.extracted_amount) > 0.01, TRUE, FALSE) AS amount_mismatch
FROM
  `fraud_data.historical_transactions` t
JOIN
  ExtractedData e
ON
  t.receipt_image_gcs_path = e.uri;